# Day 2 | Lab 2.3: Advanced Prompting — Zero-Shot, Few-Shot, CoT, ToT, CoVe

**Duration:** ~1.5 hours

**Scenario.** Retail + corporate banking (customer support, transaction categorization, structured intake, loan assessment, debt management, RBI guideline summarization, stock analysis) — preserved from the merged GM Lab 3 + Lab 4 source notebooks.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Apply **zero-shot** and **few-shot** prompting to common banking tasks and choose between them.
2. Use **Chain-of-Thought (CoT)** to make multi-step reasoning auditable.
3. Apply a **Tree-of-Thought (ToT)-style** prompt to explore multiple hypotheses before deciding.
4. Use **Chain-of-Verification (CoVe)** to reduce factual hallucinations on document summarization.
5. Decide which technique to use given a task's complexity, stakes, and latency budget.

**Tools.** LangChain v1 · `langchain-openai` · `gpt-4.1-mini` (CoT/ToT) · `gpt-4o-mini` (CoVe).

*Merged from GM Module 7 Labs 3 & 4 · Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-community>=0.3' python-dotenv


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


In [3]:
import os
from IPython.display import display, Image, Markdown
# from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
# load_dotenv()

# Initialize the model (we use a slightly higher temperature for more creative/reasoning tasks)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)

# Helper function to print results neatly
def execute_prompt(prompt_template, title, **kwargs):
    """
    Executes a given prompt template with context variables and prints the output.
    """
    print(f"--- {title} ---")

    # Create the prompt using the template and input variables
    prompt = ChatPromptTemplate.from_template(prompt_template)

    # Create the chain
    chain = prompt | llm | StrOutputParser()

    # Invoke the chain with the provided context
    response = chain.invoke(kwargs)

    print("PROMPT:")
    print(prompt.format(**kwargs))
    print("\n" + "="*20 + " RESPONSE " + "="*20 + "\n")
    # print(response)
    display(Markdown(response))
    print("\n" + "-"*50 + "\n")

-----

## **1. Zero-Shot Prompting: Instructing Without Examples**

### **Background**

**Zero-shot prompting** is the most fundamental technique. It involves asking the model to perform a task for which it has received no specific, in-prompt examples. The model relies entirely on its pre-existing knowledge to understand the instruction and generate a response.

  * **How it works:** You give a direct command. For example, "Translate this text to French." The model has seen countless translation tasks during its training and knows what to do.
  * **Strengths:** Simple, fast, and effective for straightforward tasks that fall within the model's general capabilities (e.g., summarization, general classification, simple Q\&A).
  * **Weaknesses:** It can struggle with novel, highly specific, or nuanced tasks where the desired output format or logic is not obvious.
  * **Banking Use Cases:** Initial customer feedback analysis, categorizing support tickets into broad topics, drafting simple emails, identifying the general intent of a customer query.

### **Use Case 1.1: Sentiment Analysis of Bank Feedback**

**Scenario:** A bank's automated system collects feedback from a customer after a service call. We need to classify the sentiment.

#### **Naive Prompt 👎**

This prompt is too simple and gives the model no guidance on the output format or classification scheme.

In [ ]:
customer_feedback_1 = "The mobile app is really slow to load my account balance, and it crashed twice yesterday. I'm very frustrated."

naive_prompt_1 = "{feedback}"

execute_prompt(
    naive_prompt_1,
    "1.1 Naive Sentiment Analysis",
    feedback=customer_feedback_1
)

**Expected Response:**
The response will be a verbose rephrasing of the feedback, not a clean classification.

#### **Improved Zero-Shot Prompt 👍**

This prompt gives a clear **role**, **task**, and **strict output constraints**, guiding the model to perform a classification task accurately.

In [ ]:
customer_feedback_1 = "The mobile app is really slow to load my account balance, and it crashed twice yesterday. I'm very frustrated."

zeroshot_prompt_1 = """
You are a sentiment analysis AI for a major bank. Your task is to classify customer feedback into one of three categories: Positive, Negative, or Neutral.

Analyze the following feedback and provide only the category name as the output.

Feedback: "{feedback}"
"""

execute_prompt(
    zeroshot_prompt_1,
    "1.1 Improved Zero-Shot Sentiment Analysis",
    feedback=customer_feedback_1
)

-----

### **Use Case 1.2: Identifying Customer Intent**

**Scenario:** A chatbot needs to understand why a customer is messaging to route them to the correct department.

#### **Naive Prompt 👎**

This prompt is a vague question that will lead to a conversational and unstructured answer.

In [ ]:
customer_query_1 = "Hi, I can't seem to find my last month's credit card statement in the app."

naive_prompt_2 = "What does this customer want to do? Query: {query}"

# execute_prompt(
#     naive_prompt_2,
#     "1.2 Naive Intent Recognition",
#     query=customer_query_1
# )

#### **Improved Zero-Shot Prompt 👍**

This prompt defines a set of valid intents, forcing the model to classify the query into a predefined category suitable for automated routing.

In [ ]:
customer_query_1 = "Hi, I can't seem to find my last month's credit card statement in the app."

zeroshot_prompt_2 = """
You are an intent recognition model for a banking chatbot. Classify the user's query into one of the following predefined categories:
- Card Services
- Account Inquiry
- Loan Application
- Technical Support
- General Question

User Query: "{query}"

Return only the category name.
"""

# execute_prompt(
#     zeroshot_prompt_2,
#     "1.2 Improved Zero-Shot Intent Recognition",
#     query=customer_query_1
# )

-----

## **2. Few-Shot Prompting: Learning from Examples**

### **Background**

**Few-shot prompting** involves providing the LLM with a few examples ("shots") of the task you want it to perform directly within the prompt. This helps the model understand the desired pattern, format, and nuances.

  * **How it works:** You show the model input-output pairs. For instance, `Input: Sea otter -> Output: Enhydra lutris. Input: Manatee -> Output: Trichechus. Input: Blue whale -> Output:`. The model learns the pattern (common name to scientific name) and completes it.
  * **Strengths:** Extremely effective for tasks requiring a specific output structure (like JSON), a particular style, or classification into custom, non-obvious categories.
  * **Weaknesses:** Requires crafting high-quality examples, and makes the prompt longer and more expensive to process.
  * **Banking Use Cases:** Extracting specific data from unstructured documents (KYC, invoices), categorizing transactions into bank-specific buckets, formatting data for API calls, drafting communications that match a brand's voice.

### **Use Case 2.1: Custom Transaction Categorization**

**Scenario:** A bank's personal finance management tool needs to categorize transactions based on its own unique category list.

#### **Naive Prompt 👎**

The model will use its own general knowledge of categories, which won't match the bank's internal system.

In [ ]:
transaction_description_1 = "ZOMATO-FOOD-DELIVERY GURGAON"

naive_prompt_3 = "Categorize this transaction: '{transaction}'"

# execute_prompt(
#     naive_prompt_3,
#     "2.1 Naive Transaction Categorization",
#     transaction=transaction_description_1
# )

This is a good guess, but it may not match our internal category `Food & Entertainment`.

#### **Few-Shot Prompt 👍**

By providing examples, we teach the model our exact categorization schema.

In [ ]:
transaction_description_1 = "ZOMATO-FOOD-DELIVERY GURGAON"

fewshot_prompt_1 = """
You are a transaction categorization engine for a bank. Your goal is to classify transactions into one of the following categories: 'Utilities', 'Groceries', 'Travel', 'Food & Entertainment', 'Shopping', 'Miscellaneous'.

Here are some examples:
1. Transaction: "AMAZON-PAY-ELECTRICITY"
   Category: Utilities
2. Transaction: "UBER-TRIP-MUMBAI-AIRPORT"
   Category: Travel
3. Transaction: "BIGBASKET-ONLINE-GROC"
   Category: Groceries

Now, categorize the following transaction. Provide only the category name.

Transaction: "{transaction}"
Category:
"""

# execute_prompt(
#     fewshot_prompt_1,
#     "2.1 Few-Shot Transaction Categorization",
#     transaction=transaction_description_1
# )

### **Use Case 2.2: Extracting Structured Data from Text**

**Scenario:** A wealth manager receives an email from a potential client. We need to extract key details into a structured JSON object for the CRM system.

#### **Naive Prompt 👎**

This will produce an unpredictable summary, not a clean, machine-readable object.

In [ ]:
client_email_1 = "Hi, my name is Rohan Gupta. I'm 42 years old and I'm interested in your wealth management services. My investment portfolio is currently around $500,000 and my risk tolerance is moderately aggressive. You can reach me at rohan.gupta@email.com."

naive_prompt_4 = "Extract the client details from this email: {email}"

# execute_prompt(
#     naive_prompt_4,
#     "2.2 Naive Data Extraction",
#     email=client_email_1
# )

#### **Few-Shot Prompt 👍**

The example teaches the model the exact JSON schema we require.

In [ ]:
client_email_1 = "Hi, my name is Rohan Gupta. I'm 42 years old and I'm interested in your wealth management services. My investment portfolio is currently around $500,000 and my risk tolerance is moderately aggressive. You can reach me at rohan.gupta@email.com."

fewshot_prompt_2 = """
You are a data extraction AI that converts unstructured client emails into structured JSON objects.

---
Example 1:
Email: "My name is Priya Singh, my risk appetite is conservative and my portfolio is $1.2M. I am 55."
JSON:
{{
  "name": "Priya Singh",
  "age": 55,
  "portfolio_value": 1200000,
  "risk_profile": "Conservative",
  "contact_email": "N/A"
}}
---

Now, process the following email and provide only the JSON object as output.

Email: "{email}"
JSON:
"""

# execute_prompt(
#     fewshot_prompt_2,
#     "2.2 Few-Shot Data Extraction",
#     email=client_email_1
# )

-----

## **3. Chain-of-Thought (CoT) Prompting: Showing Your Work**

### **Background**

**Chain-of-Thought (CoT) prompting** is a technique that encourages the model to break down a complex problem into intermediate steps before giving a final answer. By simply adding phrases like "Let's think step by step," you can significantly improve performance on tasks that require logic, reasoning, or calculation.

  * **How it works:** Instead of jumping to a conclusion, the model first "thinks" out loud, detailing its reasoning process. This mimics how humans solve complex problems and often leads the model to a more accurate result.
  * **Strengths:** Massively improves accuracy on arithmetic, commonsense, and symbolic reasoning tasks. The output is more transparent and auditable, as you can see *how* the model reached its conclusion.
  * **Banking Use Cases:** Assessing loan eligibility based on multiple criteria, preliminary fraud analysis, calculating financial ratios, explaining complex financial products to customers, and checking compliance with internal policies.

### **Use Case 3.1: Preliminary Fraud Alert Analysis**

**Scenario:** A transaction is flagged for potential fraud. A junior analyst needs a summary explaining why it's suspicious.

#### **Naive Prompt 👎**

This prompt asks for a simple "yes/no" answer, which is not helpful and lacks any justification.

In [4]:
transaction_data_1 = """
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.
"""

naive_prompt_5 = "Is this transaction likely fraudulent? Transaction Details: {transaction}"

execute_prompt(
    naive_prompt_5,
    "3.1 Naive Fraud Analysis",
    transaction=transaction_data_1
)

--- 3.1 Naive Fraud Analysis ---
PROMPT:
Human: Is this transaction likely fraudulent? Transaction Details: 
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.


==================== RESPONSE ====================



Based on the details provided, this transaction appears suspicious and potentially fraudulent due to several factors:

1. **Unusual Location**: The customer, Anjali Rao, is usually located in Bangalore, India, but this transaction occurred in Prague, Czech Republic. There is no prior history of international transactions.

2. **High Transaction Amount**: The transaction amount is ₹85,000, which is significantly higher than the customer's average transaction value of ₹2,500.

3. **Time of Transaction**: The transaction took place at 3:15 AM IST, which is an unusual time for a customer to be making purchases, especially internationally.

4. **Merchant Category**: The purchase is from a luxury watches merchant category, which is a high-value and potentially high-risk category.

Given these points, it is advisable to flag this transaction for further verification with the customer before approval to prevent potential fraud.


--------------------------------------------------



#### **Chain-of-Thought Prompt 👍**

We explicitly ask the model to reason through the evidence step-by-step, providing a clear, auditable analysis.

In [5]:
transaction_data_1 = """
- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.
"""

cot_prompt_1 = """
You are a fraud detection analyst.
Analyze the following transaction details to determine if it is likely fraudulent.
Let's think step by step to build your analysis before making a final conclusion.

Transaction Details:
{transaction}

First, identify all the risk factors. For each factor, explain why it is suspicious.
Second, summarize your findings.
Finally, conclude with a risk assessment: LOW, MEDIUM, or HIGH.
"""

execute_prompt(
    cot_prompt_1,
    "3.1 Chain-of-Thought Fraud Analysis",
    transaction=transaction_data_1
)

--- 3.1 Chain-of-Thought Fraud Analysis ---
PROMPT:
Human: 
You are a fraud detection analyst. 
Analyze the following transaction details to determine if it is likely fraudulent.
Let's think step by step to build your analysis before making a final conclusion.

Transaction Details:

- Customer: Anjali Rao
- Usual Location: Bangalore, IN
- Transaction Amount: ₹85,000
- Transaction Location: Prague, CZ
- Time: 3:15 AM IST
- Merchant Category: Luxury Watches
- Customer History: No prior international transactions. Average transaction value is ₹2,500.


First, identify all the risk factors. For each factor, explain why it is suspicious.
Second, summarize your findings.
Finally, conclude with a risk assessment: LOW, MEDIUM, or HIGH.


==================== RESPONSE ====================



Let's analyze the transaction step by step.

---

### Step 1: Identify Risk Factors and Explain Why They Are Suspicious

1. **Transaction Location: Prague, CZ**  
   - The customer, Anjali Rao, is based in Bangalore, India, and has no prior history of international transactions.  
   - A sudden transaction in a foreign country, especially one where the customer has no previous activity, is suspicious and could indicate unauthorized use.

2. **Transaction Amount: ₹85,000**  
   - The average transaction value for the customer is ₹2,500, so this transaction amount is more than 30 times higher than usual.  
   - A large deviation in transaction amount is a common indicator of potential fraud.

3. **Transaction Time: 3:15 AM IST**  
   - The transaction occurred at an unusual hour for the customer’s timezone (early morning hours).  
   - Fraudulent transactions often happen at odd hours to avoid immediate detection.

4. **Merchant Category: Luxury Watches**  
   - Luxury goods are high-value items that are often targeted in fraudulent purchases due to their resale value.  
   - The customer’s usual spending pattern does not indicate purchases in luxury categories.

5. **No Prior International Transactions**  
   - The customer has no history of international spending, which makes this transaction an outlier.  
   - Fraudsters often make international purchases to exploit less familiar monitoring systems.

---

### Step 2: Summary of Findings

- The transaction is in a foreign country where the customer has no prior activity.  
- The amount is significantly higher than the customer’s usual spending.  
- The purchase time is unusual for the customer’s timezone.  
- The merchant category is for luxury goods, which is inconsistent with the customer’s typical spending behavior.  
- No prior international transactions increase the likelihood that this is an unauthorized transaction.

---

### Step 3: Risk Assessment

**Risk Level: HIGH**

Given the multiple red flags—unusual location, high transaction amount, odd timing, luxury merchant category, and no prior international history—this transaction is highly likely to be fraudulent and should be flagged for immediate review or blocking.

---

**Final Recommendation:**  
Flag the transaction for further investigation and contact the customer to verify the legitimacy of the transaction. Consider temporarily suspending the account or transaction until confirmation is obtained.


--------------------------------------------------




### **3.2. Chain-of-Thought (CoT): Evaluating a Small Business Loan**

**Scenario:** A loan officer needs a quick, reasoned assessment of a small business loan application to decide if it warrants a full underwriting process.

In [ ]:
loan_application_data = """
- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720
"""

#### **Naive Prompt 👎**

A direct question will yield a "black box" answer with minimal justification, making it hard to trust or audit.

In [ ]:
naive_prompt_3 = "Based on the data and criteria, should we approve a loan for Global Exports Inc.? \n\n{data}"

# execute_prompt(
#     naive_prompt_3,
#     "3.1 Naive Loan Assessment",
#     data=loan_application_data
# )


*Critique:* The answer is correct, but it lacks a structured breakdown of the decision process.

#### **Improved CoT Prompt 👍**

This prompt forces the model to perform a step-by-step analysis, comparing each application data point against the criteria before reaching a conclusion.

In [ ]:
cot_prompt_3 = """
You are a junior loan underwriter AI. Evaluate the small business loan application against the bank's criteria. Let's think step by step to create a clear evaluation report.

Application Data and Criteria:
{data}

**Evaluation Process:**
1.  **Check Years in Business:** Compare the applicant's years in business against the minimum requirement. State Pass or Fail.
2.  **Check Annual Revenue:** Compare the applicant's revenue against the minimum requirement. State Pass or Fail.
3.  **Check CIBIL Score:** Compare the owner's CIBIL score against the minimum requirement. State Pass or Fail.
4.  **Summary of Findings:** Briefly list the strengths and weaknesses of the application.
5.  **Final Recommendation:** Conclude with 'Proceed to Underwriting' or 'Reject'.
"""

# execute_prompt(
#     cot_prompt_3,
#     "3.2 CoT Loan Assessment",
#     data=loan_application_data
# )


*Critique:* The response is transparent, auditable, and provides a clear, well-supported rationale for the decision.


-----

## **4. Tree-of-Thought (ToT) Prompting: Exploring Multiple Paths**

### **Background**

**Tree-of-Thought (ToT)** is an advanced technique that generalizes Chain-of-Thought. Instead of just one linear reasoning path, ToT encourages the model to explore *multiple* different lines of reasoning (branches of a tree). It can then self-evaluate these paths and choose the most promising one to pursue.

  * **How it works:** A full ToT implementation is complex and often requires an agentic framework with multiple LLM calls. However, we can simulate the *spirit* of ToT in a single prompt by asking the model to:
    1.  **Generate** multiple potential solutions or viewpoints.
    2.  **Evaluate** the pros and cons of each.
    3.  **Select** the best option based on that evaluation.
  * **Strengths:** Exceptionally powerful for complex problems with no single right answer, such as strategic planning, creative problem-solving, and generating robust recommendations.
  * **Banking Use Cases:** Developing personalized financial plans for clients, creating customer retention strategies, brainstorming new product features, and formulating responses to complex regulatory inquiries.

### **Use Case 4.1: Proposing a Client Investment Strategy**

**Scenario:** A financial advisor needs to propose an investment strategy for a young client with a moderate risk appetite.

#### **Naive Prompt 👎**

This gives a generic, one-size-fits-all answer without considering alternatives or trade-offs.

In [6]:
client_profile_1 = """
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000
"""

naive_prompt_6 = "Suggest an investment portfolio for this client: {profile}"

execute_prompt(
    naive_prompt_6,
    "4.1 Naive Investment Suggestion",
    profile=client_profile_1
)

--- 4.1 Naive Investment Suggestion ---
PROMPT:
Human: Suggest an investment portfolio for this client: 
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000


==================== RESPONSE ====================



Certainly! For Sameer Khan, age 28, aiming for early retirement with a moderate risk tolerance and a healthy annual income of ₹25,00,000, the investment portfolio should balance growth potential with risk management. Given his relatively young age, he can still take advantage of compounding by investing significantly in growth assets but should also include some stability to match his moderate risk profile.

---

### Investment Portfolio Suggestion for Sameer Khan

**1. Equity (60%) – Growth Focus**
- **Large-cap Mutual Funds / ETFs (30%)**  
  Stable companies with consistent performance. Examples: SBI Bluechip Fund, HDFC Top 100 Fund.
- **Mid-cap / Small-cap Mutual Funds (20%)**  
  Higher growth potential but more volatile. Examples: DSP Midcap Fund, Axis Small Cap Fund.
- **Direct Equity (10%)**  
  Select high-quality stocks from sectors with strong growth potential (technology, FMCG, pharma).

**2. Debt Instruments (25%) – Stability and Income**
- **Corporate Bond Funds / Debt Mutual Funds (15%)**  
  Lower risk than equities, provide steady returns.
- **Public Provident Fund (PPF) / Fixed Deposits (10%)**  
  Safe, tax-efficient, and good for long-term stability.

**3. Hybrid / Balanced Funds (10%)**  
Funds that invest in a mix of equity and debt, suitable for moderate risk tolerance. Examples: ICICI Prudential Balanced Advantage Fund.

**4. Alternative Investments (5%)**  
- **Gold ETFs / Sovereign Gold Bonds**  
  Hedge against inflation and diversify portfolio.

---

### Additional Recommendations:
- **Systematic Investment Plan (SIP):** Invest monthly to benefit from rupee cost averaging.
- **Emergency Fund:** Keep 6-12 months of expenses in a liquid savings or money market fund.
- **Tax Planning:** Utilize Section 80C (PPF, ELSS funds) to save taxes.
- **Review Portfolio Annually:** Rebalance to maintain desired asset allocation.

---

### Sample Allocation (Based on ₹25,00,000 Annual Income)

| Asset Class            | % Allocation | Annual Investment (₹) | Notes                          |
|-----------------------|--------------|----------------------|--------------------------------|
| Large-cap Equity Funds | 30%          | ₹7,50,000            | Stable growth                  |
| Mid/Small-cap Funds   | 20%          | ₹5,00,000            | Higher growth potential         |
| Direct Equity         | 10%          | ₹2,50,000            | Select stocks                  |
| Debt Funds            | 15%          | ₹3,75,000            | Corporate bonds, debt funds    |
| PPF / Fixed Deposits  | 10%          | ₹2,50,000            | Safe, tax-efficient            |
| Hybrid Funds          | 10%          | ₹2,50,000            | Balanced risk                  |
| Gold ETFs / Sovereign Gold Bonds | 5% | ₹1,25,000            | Inflation hedge, diversification |

---

If Sameer wants to retire early (say by 45-50), he should aim to maximize his savings rate and invest aggressively in equities early on, gradually shifting to safer assets as he approaches retirement.

Would you like me to help with specific fund recommendations or a retirement corpus calculation?


--------------------------------------------------



#### **"ToT-style" Prompt 👍**

This prompt guides the model to explore and critique several strategies before making a final, well-justified recommendation.

In [7]:
client_profile_1 = """
- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000
"""

tot_prompt_1 = """
You are a senior investment strategist. Your task is to devise an investment strategy for a new client.
Use the following structured thinking process:

Client Profile:
{profile}

1.  **Propose Three Distinct Strategies:** Based on the client's profile, generate three different investment strategies. Name each one (e.g., 'Balanced Growth', 'Aggressive Tech Focus', 'Diversified Core'). For each, suggest a sample asset allocation (Equities, Debt, Alternatives).
2.  **Evaluate Each Strategy:** For each of the three strategies, briefly list the potential Pros and Cons specifically for this client. Consider their age, goal, and risk tolerance.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy for the client and write a concluding paragraph explaining why it is the best choice.
"""

execute_prompt(
    tot_prompt_1,
    "4.1 ToT-style Investment Strategy",
    profile=client_profile_1
)

--- 4.1 ToT-style Investment Strategy ---
PROMPT:
Human: 
You are a senior investment strategist. Your task is to devise an investment strategy for a new client.
Use the following structured thinking process:

Client Profile:

- Name: Sameer Khan
- Age: 28
- Goal: Wealth accumulation for early retirement
- Risk Tolerance: Moderate
- Annual Income: ₹25,00,000


1.  **Propose Three Distinct Strategies:** Based on the client's profile, generate three different investment strategies. Name each one (e.g., 'Balanced Growth', 'Aggressive Tech Focus', 'Diversified Core'). For each, suggest a sample asset allocation (Equities, Debt, Alternatives).
2.  **Evaluate Each Strategy:** For each of the three strategies, briefly list the potential Pros and Cons specifically for this client. Consider their age, goal, and risk tolerance.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy for the client and write a concluding paragraph explaining why it is the

Certainly! Below is a structured investment strategy proposal for Sameer Khan based on his profile:

---

### Client Profile Recap:
- **Name:** Sameer Khan  
- **Age:** 28  
- **Goal:** Wealth accumulation for early retirement (likely 10-15 years horizon)  
- **Risk Tolerance:** Moderate  
- **Annual Income:** ₹25,00,000  

---

## 1. Propose Three Distinct Strategies

### Strategy 1: Balanced Growth  
- **Equities:** 60% (Mix of large-cap, mid-cap, and some international equities)  
- **Debt:** 30% (Government bonds, high-quality corporate bonds, fixed deposits)  
- **Alternatives:** 10% (REITs, gold ETFs, and some exposure to private equity or venture funds)  

### Strategy 2: Aggressive Tech Focus  
- **Equities:** 80% (Heavy focus on technology, innovation-driven sectors, and emerging markets)  
- **Debt:** 10% (Short-term debt instruments for liquidity)  
- **Alternatives:** 10% (Cryptocurrency, thematic funds, and venture capital exposure)  

### Strategy 3: Diversified Core  
- **Equities:** 50% (Broad market index funds, large-cap stocks, and dividend-paying stocks)  
- **Debt:** 40% (Mix of government securities, corporate bonds, and fixed income mutual funds)  
- **Alternatives:** 10% (Gold, real estate funds, and infrastructure investments)  

---

## 2. Evaluate Each Strategy

### Strategy 1: Balanced Growth  
- **Pros:**  
  - Aligns well with moderate risk tolerance.  
  - Provides growth potential through equities balanced with stability from debt.  
  - Alternatives add diversification and inflation hedge.  
- **Cons:**  
  - May not maximize returns aggressively enough for early retirement goal.  
  - Moderate exposure to alternatives might limit upside.  

### Strategy 2: Aggressive Tech Focus  
- **Pros:**  
  - High growth potential suitable for a young investor.  
  - Exposure to innovation sectors can lead to outsized returns.  
- **Cons:**  
  - High volatility and risk, which may not suit moderate risk tolerance.  
  - Concentration risk in tech and emerging markets.  
  - Limited debt allocation reduces downside protection.  

### Strategy 3: Diversified Core  
- **Pros:**  
  - Strong capital preservation with higher debt allocation.  
  - Steady income from dividend stocks and bonds.  
  - Good diversification reduces portfolio volatility.  
- **Cons:**  
  - Lower equity exposure may slow wealth accumulation.  
  - May not achieve early retirement target as quickly.  

---

## 3. Recommend and Justify

**Recommended Strategy: Balanced Growth**

Given Sameer’s age (28) and goal of early retirement, he has a relatively long investment horizon which allows for meaningful equity exposure to drive wealth accumulation. However, his moderate risk tolerance suggests that an overly aggressive portfolio could cause undue stress and potential premature liquidation during market downturns. The Balanced Growth strategy strikes an optimal balance by allocating a significant portion to equities for growth, while maintaining a meaningful allocation to debt to reduce volatility and provide stability. The inclusion of alternatives like REITs and gold ETFs adds diversification and inflation protection, which is important for long-term wealth preservation. This approach aligns well with his income level and goal, providing a disciplined yet growth-oriented path to early retirement.

---

If you would like, I can also provide specific fund or stock recommendations within these allocations or a suggested savings and SIP plan.


--------------------------------------------------




-----

### **4.2. Tree-of-Thought (ToT): Devising a Debt Restructuring Plan**

**Scenario:** A financial advisor is helping a client who is overwhelmed with multiple sources of debt. The goal is to propose and compare different concrete strategies for tackling the debt.

In [9]:
client_debt_profile = """
- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR
"""

#### **Naive Prompt 👎**

This prompt will generate generic, non-comparative advice that isn't tailored into actionable, alternative strategies.

In [10]:
naive_prompt_4 = "How can this client manage their debt? \n\n{profile}"

execute_prompt(
    naive_prompt_4,
    "4.1 Naive Debt Advice",
    profile=client_debt_profile
)

--- 4.1 Naive Debt Advice ---
PROMPT:
Human: How can this client manage their debt? 


- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR


==================== RESPONSE ====================



To help Priya manage her debt effectively, the goal should be to minimize interest payments and pay off high-interest debt first while maintaining manageable monthly payments. Here's a step-by-step approach:

---

### 1. **Assess the Debt Situation**

| Debt Type      | Principal (₹) | APR (%) | Monthly Interest Approx. (₹) |
|----------------|---------------|---------|------------------------------|
| Credit Card    | 2,00,000      | 36%     | 6,000                        |
| Personal Loan  | 5,00,000      | 14%     | 5,833                        |
| Car Loan       | 3,00,000      | 9%      | 2,250                        |

*Note: Monthly interest approx. = (Principal × APR) / 12*

---

### 2. **Prioritize Debt Repayment**

- **Highest priority:** Credit Card (36% APR) — extremely high interest.
- **Second priority:** Personal Loan (14% APR).
- **Last priority:** Car Loan (9% APR).

---

### 3. **Debt Repayment Strategy**

- **Available for debt repayment:** ₹35,000/month.
- **Minimum payments:** Ensure minimum payments on all debts to avoid penalties.
- **Extra payments:** Apply any surplus to the highest interest debt.

---

### 4. **Suggested Monthly Payment Allocation**

| Debt Type      | Minimum Payment (Estimate) | Extra Payment | Total Payment |
|----------------|----------------------------|---------------|---------------|
| Credit Card    | ₹10,000                    | ₹15,000       | ₹25,000       |
| Personal Loan  | ₹7,000                     | ₹3,000        | ₹10,000       |
| Car Loan       | ₹5,000                     | ₹0            | ₹5,000        |

*Note: These are approximate minimum payments; actual minimums may vary.*

- Pay **₹25,000** to credit card to aggressively reduce this high-interest debt.
- Pay **₹10,000** to personal loan to maintain steady progress.
- Pay minimum **₹5,000** to car loan to keep it current.

---

### 5. **Additional Recommendations**

- **Avoid new credit card debt:** Stop using the credit card until it’s paid off.
- **Consider balance transfer:** If possible, transfer credit card balance to a lower-interest loan or card with a 0% introductory offer.
- **Emergency fund:** Maintain a small emergency fund (₹20,000-₹30,000) to avoid new debt.
- **Refinance personal or car loan:** If possible, look for lower interest rates to reduce monthly payments.
- **Track expenses:** Continue monitoring expenses to ensure ₹45,000 is sustainable and look for further savings.

---

### 6. **Projected Outcome**

- Aggressively paying off credit card debt first will reduce the high interest burden.
- Once credit card is paid off (approx. 8-9 months), redirect that ₹25,000 payment toward the personal loan to accelerate repayment.
- After personal loan is cleared, focus all payments on the car loan.

---

### Summary

| Step | Action                              |
|-------|-----------------------------------|
| 1     | Pay off credit card aggressively  |
| 2     | Maintain minimum payments on others |
| 3     | Avoid new debt                    |
| 4     | Consider refinancing options      |
| 5     | Build emergency fund              |

This approach will minimize interest paid and help Priya become debt-free faster while staying within her budget.


--------------------------------------------------



*Critique:* The advice is sound but lacks structure and fails to explore or compare alternative, well-known strategies.

#### **Improved "ToT-style" Prompt 👍**

This prompt instructs the model to generate and evaluate multiple distinct strategies, presenting a comprehensive analysis of the options.

In [11]:
tot_prompt_4 = """
You are an expert financial advisor. For the client profile below, devise a debt management plan using the following structured thinking process.

Client Profile:
{profile}

1.  **Propose Three Distinct Strategies:** Generate three different debt repayment strategies. Name each one (e.g., 'Debt Avalanche', 'Debt Snowball', 'Loan Consolidation'). Briefly explain how each would work for this client.
2.  **Evaluate Each Strategy:** For each of the three strategies, list the primary 'Pro' (e.g., lowest total interest paid) and the primary 'Con' (e.g., may feel slower) for this specific client.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy and explain why it is the best recommendation for someone in this situation.
"""

execute_prompt(
    tot_prompt_4,
    "4.2 ToT-style Debt Plan",
    profile=client_debt_profile
)

--- 4.2 ToT-style Debt Plan ---
PROMPT:
Human: 
You are an expert financial advisor. For the client profile below, devise a debt management plan using the following structured thinking process.

Client Profile:

- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR


1.  **Propose Three Distinct Strategies:** Generate three different debt repayment strategies. Name each one (e.g., 'Debt Avalanche', 'Debt Snowball', 'Loan Consolidation'). Briefly explain how each would work for this client.
2.  **Evaluate Each Strategy:** For each of the three strategies, list the primary 'Pro' (e.g., lowest total interest paid) and the primary 'Con' (e.g., may feel slower) for this specific client.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy 

Certainly! Let’s analyze Priya’s debt situation and devise a structured debt management plan.

---

### Client Summary:
- **Monthly Take-home Salary:** ₹80,000  
- **Monthly Expenses (non-debt):** ₹45,000  
- **Available for Debt Repayment:** ₹35,000  
- **Debts:**  
  1. Credit Card: ₹2,00,000 @ 36% APR  
  2. Personal Loan: ₹5,00,000 @ 14% APR  
  3. Car Loan: ₹3,00,000 @ 9% APR  

---

## 1. Propose Three Distinct Strategies

### Strategy A: Debt Avalanche  
- **How it works:** Prioritize paying off the debt with the highest interest rate first while making minimum payments on the others.  
- **For Priya:** Pay minimum on Personal Loan and Car Loan, and allocate the maximum possible (₹35,000) to the Credit Card debt first (36% APR). Once Credit Card is paid off, move to Personal Loan (14%), then Car Loan (9%).  

### Strategy B: Debt Snowball  
- **How it works:** Prioritize paying off the smallest debt first to build momentum and motivation, regardless of interest rate.  
- **For Priya:** Pay minimum on Personal Loan and Credit Card, allocate maximum to Car Loan (₹3,00,000) first (smallest balance), then move to Credit Card, then Personal Loan.  

### Strategy C: Loan Consolidation / Refinancing  
- **How it works:** Combine high-interest debts into a single loan with a lower interest rate, reducing monthly interest and simplifying payments.  
- **For Priya:** Explore options to refinance Credit Card and Personal Loan into a lower-interest personal loan or balance transfer credit card with a 0% or low introductory rate. Then pay ₹35,000 monthly towards this consolidated debt. Car loan remains separate or included if possible.  

---

## 2. Evaluate Each Strategy

| Strategy           | Primary Pro                                         | Primary Con                                         |
|--------------------|---------------------------------------------------|----------------------------------------------------|
| **Debt Avalanche** | Minimizes total interest paid, fastest payoff overall | Credit Card debt may feel overwhelming due to high balance and interest, slower psychological wins |
| **Debt Snowball**  | Quick wins by clearing smaller debts first, boosts motivation | Higher total interest paid, especially on costly Credit Card debt |
| **Loan Consolidation** | Potentially lowers interest rates and monthly payments, simplifies management | Requires good credit score and access to refinancing; may involve fees or not be feasible |

---

## 3. Recommend and Justify

### Recommended Strategy: **Debt Avalanche**

**Justification:**  
- Priya’s highest interest debt is the Credit Card at 36% APR, which is extremely costly. Prioritizing this will save her the most money in interest payments over time.  
- Although the Credit Card balance is ₹2,00,000 (not the smallest), aggressively paying it down first with ₹35,000/month will reduce the compounding interest burden quickly.  
- Once the Credit Card is cleared, she can focus on the Personal Loan and then the Car Loan, both of which have significantly lower interest rates.  
- This strategy is financially optimal given her available repayment capacity and the high cost of the Credit Card debt.  
- While the Debt Snowball approach offers psychological benefits, the high interest on the Credit Card means delaying its payoff will cost her significantly more.  
- Loan consolidation is worth exploring but depends on her creditworthiness and market options; it may not be immediately available or cost-effective.

---

### Additional Recommendations:

- **Avoid accumulating new credit card debt** during repayment.  
- **Consider negotiating with lenders** for lower interest rates or restructuring if possible.  
- **Build an emergency fund** once debts are under control to avoid future reliance on high-interest credit.

---

If Priya follows the Debt Avalanche strategy, she will minimize interest costs and become debt-free faster, putting her in a stronger financial position.


--------------------------------------------------




*Critique:* This response provides a strategic, multi-faceted analysis that empowers the client and advisor to make an informed decision based on clear trade-offs.

---

## Part 2 — Chain-of-Verification (CoVe)

We've seen Zero-Shot, Few-Shot, CoT, and ToT-style prompting. Now we add a **verification step** — the model double-checks its own draft against the source before producing the final answer. Empirically, CoVe substantially reduces hallucinations on document summarization and factual analysis.

**The 4-step CoVe pattern:**
1. **Draft** — generate an initial response.
2. **Plan verification questions** — what facts does the draft claim?
3. **Answer each question independently** — using only the source.
4. **Revise** — produce a final response consistent with the verified answers.


-----

## **Chain-of-Verification (CoVe): Minimizing Hallucinations**

### **Background**

Factual accuracy is non-negotiable in banking. **Chain-of-Verification (CoVe)** is a technique designed to make an LLM double-check its own work, significantly reducing the risk of generating plausible-sounding but incorrect information (hallucinations).

  * **How it works:** It's a deliberate, multi-step process.
    1.  **Draft Response:** The LLM generates an initial answer to the query.
    2.  **Plan Verifications:** The model reviews its own draft and generates a list of factual claims that need to be checked.
    3.  **Execute Verifications:** The model answers these verification questions *independently*, which prevents the initial draft from biasing the check.
    4.  **Final Verified Response:** The model revises the initial draft based on the outcome of the verifications, correcting any inaccuracies.
  * **Strengths:** Drastically improves the factual grounding and reliability of generated text. It's an essential technique for high-stakes content generation.
  * **Banking Use Cases:** Summarizing complex regulatory documents (like RBI circulars), generating financial reports, creating internal policy documentation, or drafting market analysis where every claim must be backed by data.

### **Use Case 1: Summarizing a Regulatory Guideline**

**Scenario:** A compliance officer needs a quick, accurate summary of a new (fictional) RBI guideline on digital lending.

In [4]:
rbi_guideline_text = """
RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.
"""

#### **Naive Prompt 👎**

A simple summarization request can lead to subtle but critical errors of omission or misinterpretation.

In [ ]:
naive_prompt_cove_1 = "Summarize the key points of the following RBI guideline: \n\n{document}"
naive_summary = execute_prompt(
    naive_prompt_cove_1,
    "2.1 Naive Guideline Summary",
    document=rbi_guideline_text
)

--- 2.1 Naive Guideline Summary ---
PROMPT:
Human: Summarize the key points of the following RBI guideline: 


RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.


==================== RESPONSE ====================

The RBI Circular No. 2025/11/A-34 outlines guidelines for Non-Banking Financial Companies (NBFCs) inv


**Critique:** The summary is mostly correct, but it makes one critical mistake: it claims the rules apply to "all financial institutions," when the text clearly exempts Regional Rural Banks (RRBs).

#### **CoVe Implementation 👍**

We simulate the CoVe process using a sequence of prompts to force self-correction.

In [ ]:
# --- CoVe Step 1: Draft Initial Response ---
step1_prompt = "Draft a summary of the key points of the following RBI guideline: \n\n{document}"
draft_summary = execute_prompt(
    step1_prompt,
    "CoVe Step 1: Draft Summary",
    document=rbi_guideline_text
)

# --- CoVe Step 2: Plan Verifications ---
step2_prompt = """
Based on the following summary, generate a list of verification questions to fact-check its claims against the original document.

Summary:
{summary}
"""
verification_plan = execute_prompt(
    step2_prompt,
    "CoVe Step 2: Plan Verifications",
    summary=draft_summary
)

# --- CoVe Step 3: Execute Verifications (Simulated) ---
# In a real system, you'd answer these questions independently. Here, we bundle them.
step3_prompt = """
Answer the following questions based *only* on the provided RBI document.

Document:
{document}

Questions:
{questions}
"""
verification_results = execute_prompt(
    step3_prompt,
    "CoVe Step 3: Execute Verifications",
    document=rbi_guideline_text,
    questions=verification_plan
)

# --- CoVe Step 4: Generate Final Verified Response ---
step4_prompt = """
You have an initial draft summary and a set of verified facts.
Your task is to rewrite the summary, correcting any inaccuracies or omissions based on the verified facts.

Initial Summary:
{summary}

Verified Facts (Q&A):
{verifications}

Final Verified Summary:
"""
final_summary = execute_prompt(
    step4_prompt,
    "CoVe Step 4: Final Verified Summary",
    summary=draft_summary,
    verifications=verification_results
)

--- CoVe Step 1: Draft Summary ---
PROMPT:
Human: Draft a summary of the key points of the following RBI guideline: 


RBI Circular No. 2025/11/A-34
Subject: Guidelines on Digital Lending for Non-Banking Financial Companies (NBFCs)

1. Introduction: To ensure customer protection and data security, all NBFCs engaged in digital lending must adhere to the following.
2. Loan Disbursal: All loan disbursals must be executed directly from the NBFC's bank account to the borrower's bank account. No third-party pool accounts are permitted. This rule is effective from December 1, 2025.
3. Data Privacy: Only minimal, necessary customer data may be collected. Explicit customer consent is required for any data sharing. Biometric data cannot be stored.
4. Applicability: These guidelines apply to all NBFCs. Regional Rural Banks (RRBs) are currently exempt.


==================== RESPONSE ====================

The Reserve Bank of India (RBI) has issued Circular No. 2025/11/A-34, outlining new guideline

This final version correctly captures the nuance of the exemption, which the naive prompt missed.

### **Chain-of-Verification (CoVe) Use Case 2: Generating Factual Stock Analysis**

**Scenario:** A junior equity analyst needs to write a short, factually precise summary of a company's quarterly performance for an internal client newsletter. The summary must not contain any speculation.

In [ ]:
# Fictional quarterly data points for "Innovate Corp."
stock_data = """
- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.
"""

#### **Naive Prompt 👎**

The model might try to be "helpful" by inventing a narrative or a cause-and-effect relationship that isn't supported by the facts provided.

In [ ]:
naive_prompt_cove_2 = "Write a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: \n\n{data}"
naive_analysis = execute_prompt(
    naive_prompt_cove_2,
    "2.2 Naive Stock Analysis",
    data=stock_data
)

--- 2.2 Naive Stock Analysis ---
PROMPT:
Human: Write a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: 


- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


==================== RESPONSE ====================

Innovate Corp's latest quarterly results showcase a solid performance, with a notable 12% year-over-year increase in revenue, reaching ₹150 Crores. This growth is complemented by an 8% increase in net profit, amounting to ₹25 Crores, indicating a healthy profit margin and operational efficiency. A key highlight for the quarter was the successful launch of their new AI platform 'Cognito', signaling the company's commitment to innovation and potentially opening new revenue streams in the future. 

**Critique:** The phrase "market reacted skeptically" directly and incorrectly links the stock drop to the results, a causal relationship not present in the source data. This is speculation, not factual reporting.

#### **CoVe Implementation 👍**

The CoVe process forces the model to check its own claims against the source data, stripping out any speculative connections.

In [ ]:
# --- CoVe Step 1: Draft Initial Response ---
step1_prompt = "Draft a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: \n\n{data}"
draft_analysis = execute_prompt(
    step1_prompt,
    "CoVe Step 1: Draft Analysis",
    data=stock_data
)

# --- CoVe Step 2: Plan Verifications ---
step2_prompt = """
Based on the following draft analysis, generate a list of verification questions to fact-check its claims against the original data points.

Draft Analysis:
{analysis}
"""
verification_plan_2 = execute_prompt(
    step2_prompt,
    "CoVe Step 2: Plan Verifications",
    analysis=draft_analysis
)

# --- CoVe Step 3: Execute Verifications ---
step3_prompt = """
Answer the following questions based *only* on the provided data points. Do not infer or speculate.

Data Points:
{data}

Questions:
{questions}
"""
verification_results_2 = execute_prompt(
    step3_prompt,
    "CoVe Step 3: Execute Verifications",
    data=stock_data,
    questions=verification_plan_2
)

# --- CoVe Step 4: Generate Final Verified Response ---
step4_prompt = """
You have a draft analysis and a set of verified facts.
Your task is to rewrite the analysis, ensuring every statement is directly supported by the verified facts.
Remove any speculation or causal links that are not explicitly mentioned in the data.

Draft Analysis:
{analysis}

Verified Facts (Q&A):
{verifications}

Final Verified Analysis:
"""
final_analysis = execute_prompt(
    step4_prompt,
    "CoVe Step 4: Final Verified Analysis",
    analysis=draft_analysis,
    verifications=verification_results_2
)

--- CoVe Step 1: Draft Analysis ---
PROMPT:
Human: Draft a short paragraph analyzing Innovate Corp's latest quarterly results based on these data points: 


- Company: Innovate Corp.
- Revenue: ₹150 Crores (an increase of 12% Year-over-Year).
- Net Profit: ₹25 Crores (an increase of 8% Year-over-Year).
- Key Highlight: Successfully launched new AI platform 'Cognito'.
- Stock Price Movement (last quarter): -5%.
- Analyst Consensus Rating: 'Hold'.


==================== RESPONSE ====================

Innovate Corp's latest quarterly results present a mixed yet promising picture of its financial health and strategic direction. The company reported a revenue of ₹150 Crores, marking a commendable 12% increase Year-over-Year (YoY), which underscores its ability to grow its top-line effectively. The net profit also saw a healthy growth of 8% YoY, amounting to ₹25 Crores, indicating improved profitability and operational efficiency. A key highlight of the quarter was the successful launch of '

This final version is factually impeccable. It presents the different data points without inventing a narrative to connect them, which is exactly what's required for unbiased financial reporting.

---
## 9. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Zero-shot vs few-shot** | Zero-shot for simple tasks on capable models; few-shot when format is unusual or accuracy matters |
| **Chain-of-Thought (CoT)** | Verbalize reasoning before answer — auditable, accurate, slower |
| **Tree-of-Thought (ToT)** | Explore multiple hypotheses in parallel and pick the best — for ambiguous, multi-strategy decisions |
| **Chain-of-Verification (CoVe)** | Draft → plan questions → answer independently → revise — substantial hallucination reduction |
| **Reasoning models change the math** | On `gpt-5-mini` / `o1` you don't need 'let's think step by step' — internal CoT happens automatically |
| **Cost / latency trade-off** | CoT is ~2× longer output; CoVe is ~3-4× total tokens. Worth it when accuracy matters more than speed |

**Next Lab:** Lab 2.4 — LangChain Output Parsers (PydanticOutputParser, XML, structured output) 📦


## 10. Stretch Exercise (Optional)

1. Take the loan-application CoT example and rerun it on `gpt-5-mini` with `reasoning_effort='medium'` — compare quality and total tokens.
2. Implement a 5-thought ToT loop that *programmatically* enumerates options, scores each with a separate LLM call, and returns the top scorer. Compare to the single-prompt ToT-style version.
3. Build a 10-document factual benchmark and measure CoVe's hallucination-reduction empirically (LLM-as-Judge for groundedness).
4. Add a 'CoVe-on-CoT' compound: verify each step of a CoT reasoning chain, not just the final answer.
5. Wire the CoVe loop into a LangGraph state machine — preview of Module 7.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. When does Chain-of-Thought help, and when does it hurt?**

*Hint:* Think latency, cost, and whether the task actually benefits from explicit reasoning.

*Answer sketch:* CoT helps for multi-step problems (math, legal reasoning, multi-criteria decisions) — measurably better accuracy. It hurts when latency/cost matter and the task is simple (classification, extraction): you pay 2-3× output tokens for no quality gain. On reasoning models like `gpt-5-mini`, manual CoT is also redundant — the model thinks internally.

---

**Q2. Difference between zero-shot, few-shot, and CoT — when to choose each?**

*Hint:* Examples vs reasoning vs neither.

*Answer sketch:* Zero-shot = just describe the task — works on capable models for common tasks. Few-shot = 2-8 input/output examples — locks unusual formats and decision boundaries. CoT = ask the model to reason step-by-step — unlocks multi-step accuracy. Stack them: few-shot CoT (examples that *show* reasoning) is the strongest pattern for hard tasks on non-reasoning models.

---

**Q3. What does Tree-of-Thought (ToT) actually mean — what is the algorithm?**

*Hint:* Beyond a linear chain of reasoning.

*Answer sketch:* ToT generates *multiple* candidate reasoning paths in parallel, evaluates each (often via a separate LLM call), and selects/expands the best. The full algorithm is search-tree-based with backtracking; the prompt-only 'ToT-style' version (this lab) just asks the model to enumerate and compare options in one shot. The full version trades cost for robustness on ambiguous decisions.

---

**Q4. Why does CoVe reduce hallucination — what's the underlying mechanism?**

*Hint:* Draft → verify → revise.

*Answer sketch:* By forcing the model to (a) plan verification questions and (b) answer each *independently* using only the source, CoVe breaks the autoregressive momentum that often produces fluent-but-wrong claims. Each verification answer is a fresh prompt with focused context, so unsupported claims don't survive the second pass.

---

**Q5. When would you skip CoVe despite the accuracy gain?**

*Hint:* Latency, cost, and use-case stakes.

*Answer sketch:* Skip when latency matters (CoVe = 3-4× round trips), when output is non-factual (creative writing), or when stakes are low (drafting boilerplate). Use it for compliance reports, regulatory summaries, financial analysis, customer-facing facts — places where a hallucination has real-world cost.

---

**Q6. How do you decide how many few-shot examples to provide?**

*Hint:* Diminishing returns vs context cost.

*Answer sketch:* Start at 2-3 (covers the common cases). Add examples that demonstrate *edge cases* the model is mishandling — not more of the same. Marginal value drops sharply after ~8 examples. Above that, switch to dynamic example selection (semantic similarity to the input) rather than adding more static examples.

---

**Q7. What's the role of 'let's think step by step' — is it still effective on newer models?**

*Hint:* Reasoning models do this internally.

*Answer sketch:* On older / smaller / non-reasoning models (Llama 3, Claude Haiku, gpt-4.1-mini), the phrase still triggers explicit CoT and improves multi-step accuracy. On reasoning models (`gpt-5-mini`, `o1`, Claude with extended thinking), it's redundant — they think internally regardless. Setting `reasoning_effort` is the right knob there.

